In [1]:
from dotenv import load_dotenv
load_dotenv() 

True

In [3]:
from pydantic import BaseModel
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from typing import Annotated 

In [4]:
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct") 

In [16]:
class ChatState(BaseModel):
    messages: Annotated[list, add_messages] 


def chatBotNode(state: ChatState) -> ChatState:
    res = llm.invoke(state.messages)
    return {
        "messages": [res]
    }

memory = InMemorySaver()

graph = StateGraph(ChatState)

graph.add_node("Node1", chatBotNode)
graph.add_edge(START, "Node1")
graph.add_edge("Node1", END)

graph = graph.compile(
    checkpointer = memory
)

res = graph.invoke(
    {"messages": [{"role": "user", "content": "Hello, My name is Dilip."}]},
    {"configurable": {"thread_id": "ChatBot123"}}
)

res["messages"][-1].content 

"Hello Dilip! It's nice to meet you. Is there something I can help you with or would you like to chat?"